# 02 – Economic Dashboard: 经济周期定位图

This notebook uses the `cycle_position` analysis module to:
1. Load US PMI and CPI data (requires FRED API key)
2. Classify each month into an economic phase (Recovery / Overheat / Stagflation / Recession)
3. Visualize the timeline and annotate the current phase

---
> **Prerequisite:** Set `FRED_API_KEY` in your `.env` file.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## Step 1 – Fetch Macro Data

In [ ]:
from src.data_fetcher.macro_economic import get_us_pmi, get_us_cpi, get_fed_funds_rate

START = "2010-01-01"

print("Fetching US PMI (NAPM) …")
pmi = get_us_pmi(start_date=START)
print(f"  {len(pmi)} months fetched")

print("Fetching US CPI (CPIAUCSL) …")
cpi = get_us_cpi(start_date=START)
print(f"  {len(cpi)} months fetched")

print("Fetching Fed Funds Rate …")
rate = get_fed_funds_rate(start_date=START)
print(f"  {len(rate)} months fetched")

## Step 2 – Compute Cycle Phases

In [ ]:
from src.analysis.cycle_position import get_cycle_position, get_current_phase

if not pmi.empty and not cpi.empty:
    cycle = get_cycle_position(pmi=pmi, cpi=cpi, rate=rate if not rate.empty else None)
    print(f"Computed {len(cycle)} monthly observations.")
    print("\nPhase distribution:")
    print(cycle["phase"].value_counts())
    
    current = get_current_phase(pmi=pmi, cpi=cpi, rate=rate if not rate.empty else None)
    print(f"\n▶ Current phase: {current['phase_label']}")
    print(f"  PMI: {current['pmi']}")
    print(f"  CPI YoY: {current['cpi_yoy']}%")
else:
    print("No PMI/CPI data available. Set FRED_API_KEY in .env.")
    cycle = None

## Step 3 – Plot Cycle Heatmap

In [ ]:
from src.visualization.dashboard_charts import plot_cycle_heatmap

if cycle is not None and not cycle.empty:
    fig = plot_cycle_heatmap(
        cycle,
        backend="plotly",
        title="US Economic Cycle: PMI vs Phase (2010–Present)",
    )
    fig.show()
else:
    print("Cycle data not available.")

## Step 4 – PMI and CPI Time-Series

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if cycle is not None and not cycle.empty:
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=["PMI (Manufacturing)", "CPI YoY (%)", "Fed Funds Rate (%)"],
        shared_xaxes=True,
        vertical_spacing=0.08,
    )
    
    fig.add_trace(go.Scatter(x=cycle.index, y=cycle["pmi"], name="PMI", line=dict(color="steelblue")), row=1, col=1)
    fig.add_hline(y=50, line_dash="dash", line_color="grey", row=1, col=1)
    
    fig.add_trace(go.Scatter(x=cycle.index, y=cycle["cpi_yoy"], name="CPI YoY", line=dict(color="tomato")), row=2, col=1)
    fig.add_hline(y=2, line_dash="dash", line_color="grey", row=2, col=1)
    
    if "rate" in cycle.columns:
        fig.add_trace(go.Scatter(x=cycle.index, y=cycle["rate"], name="Fed Funds Rate", line=dict(color="green")), row=3, col=1)
    
    fig.update_layout(height=700, title_text="US Macro Dashboard", template="plotly_white")
    fig.show()
else:
    print("Cycle data not available.")

---
**Interpretation guide:**
- 🟢 **Recovery** (PMI > 50, CPI YoY < 2%) – Growth accelerating, inflation benign
- 🟠 **Overheat** (PMI > 50, CPI YoY ≥ 2%) – Growth strong, inflation rising
- 🔴 **Stagflation** (PMI < 50, CPI YoY ≥ 2%) – Weak growth, stubborn inflation
- ⚫ **Recession** (PMI < 50, CPI YoY < 2%) – Contraction, deflationary pressure